In [ ]:
# LLM code




import sys

import json
from langchain_community.llms import Ollama
from langchain_community.chat_models import ChatOllama







for model in models:
    try:
        llm= ChatOllama(model = model, temperature=0)
        
        response = llm.invoke('When the dichromate ion is reacted, one of its most common products is Cr3+. What is the oxidation state (oxidation number) of chromium in the dichromate ion? Does reduction or oxidation occur when dichromate forms Cr3+?')
        print(f'Success:{model}')
        print(response.content)
        print('////////')
    except:
        print(f'failed:{model}-----------')




In [9]:
# text processing code 

"""
Takes in natural language, and outputs reasoning and annotations in json and natural language

"""
import sys
import os 
import json 
sys.path.append('../../src')
from pipeline.ChEBIPipeline import ChEBIPipeline as AnnotationCheck
from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
from reasoning.LLMFeedback.Pipe2NL import extract_chebi_ids, ChEBINameRetriever, replace_chebi_ids, format_json_as_prompt


def Process_Text(text):
    # start with annotations
    annotator = AnnotationCheck('/home/matt/Proj/Hermeticav2/notebooks/prototyping/QAReasoning/chebi.obo','/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl')
    annotations =annotator.process_text(text)
    print(annotations)
    Reasoned = reason_with_chebi(annotations,'/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl')
    print(Reasoned)
    # save the machine readable annotations with CHEBI IDs

    # convert the stuff back into NL for LLM input 
    finder = ChEBINameRetriever('/home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl')
    IDs = extract_chebi_ids(Reasoned)
    mapping = finder.get_chebi_names(IDs)
    PreNL = replace_chebi_ids(Reasoned,mapping)
    NL = format_json_as_prompt(PreNL)

    return NL



In [6]:
# setting up prompts


from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate

# Define your customized prompt
template = """You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a cehmistry question. The Question ans been annotated in natural language. Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It's your job to fill in LLM Response
The question is: {question}
The annotation and reasoning information is:{ChEBI_Info}

LLM Response:"""



custom_prompt = ChatPromptTemplate.from_template(template)

# Create an instance of the Ollama model with the customized prompt
ollama_model = Ollama(model="llama2")


question = "What are the electrical properties of elemental carbon?"
chebi = Process_Text(question)

p = custom_prompt.invoke({'question':question,'ChEBI_Info':chebi})

print(p)

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'What are the electrical properties of elemental carbon?', 'entities': [{'name': 'Elemental carbon', 'id': 'CHEBI:33415', 'span': (38, 54)}], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
{'text': 'What are the electrical properties of elemental carbon?', 'entities': [{'name': 'Elemental carbon', 'id': 'CHEBI:33415', 'span': (38, 54)}], 'relationships': [], 'consistency_results': [], 'reasoning': {'reasoning_steps': ['Looking up Elemental carbon (CHEBI:33415) in the ChEBI ontology', '  - Found entity in the ontology: CHEBI_33415', '  - Looking for parent classes of Elemental carbon', '    - Found parent classes: CHEBI_33259, CHEBI_50860', '  - Looking for p

In [12]:
p.messages[0].content

'You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a cehmistry question. The Question ans been annotated in natural language. Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It\'s your job to fill in LLM Response\nThe question is: What are the electrical properties of elemental carbon?\nThe annotation and reasoning information is:The original sentence analyzed is: "What are the electrical properties of elemental carbon?"\n\nThe entities identified in this sentence are: Elemental carbon (elemental carbon).\n\nThe reasoning process followed these steps: Looking up Elemental carbon (elemental carbon) in the ChEBI ontology   - Found entity in the ontology: elemental carbon   - Looking for parent classes of Elemental carbon     - Found parent classes: elemental molecular entity, organic molecular entity   - Looking for properties of Elemental

In [13]:
# Use the ollama_model to generate a response
response = ollama_model.invoke(p.messages[0].content)

print(response)

Thank you for providing the question and annotation information. Based on the provided information, I can answer the question as follows:

Elemental carbon, also known as graphite or diamond, has unique electrical properties due to its crystalline structure and bonding patterns. As an elemental molecular entity, it is a conductor of electricity in its pure form. However, when carbon is combined with other elements or substances, its electrical properties can change significantly.

Some key electrical properties of elemental carbon include:

1. Conductivity: Elemental carbon is a poor conductor of electricity, especially compared to other metals and conductive materials. This is due to the weak bonds between carbon atoms in its crystalline structure.
2. Resistivity: The resistivity of elemental carbon is relatively high, meaning it can oppose the flow of electric current. This property makes it useful in electronic devices that require low resistance, such as batteries and supercapacito

# Starting with anotations on questions 

# trying on MMLU High school Chem questions
open ended first


In [3]:
import pandas as pd
df = pd.read_parquet('/home/matt/Proj/Hermeticav2/data/Raw/MMLU/mmlu/high_school_chemistry/test-00000-of-00001.parquet')

In [4]:
df

,question,subject,choices,answer
0,London dispersion forces are caused by,high_school_chemistry,[temporary dipoles created by the position of ...,0
1,Carbon has an atomic radius of 77 pm and a fir...,high_school_chemistry,"[70 pm, 1402 kJ/mol, 86 pm, 898 kJ/mol, 135 pm...",0
2,An unknown substance is found to have a high m...,high_school_chemistry,"[ionic bonding, nonpolar covalent bonding, cov...",2
3,The net ionic equation expected when solutions...,high_school_chemistry,"[Ag+(aq) + Br-(aq) → AgBr(s), NH4+(aq) + Ag+(a...",0
4,The symbol for antimony is,high_school_chemistry,"[W, Sb, Fe, An]",1
...,...,...,...,...
198,Which of the following molecules is a strong e...,high_school_chemistry,"[CH3COOH, HC2H3O2, PCl5, HBr]",3
199,Dissolving one mole of each of the following c...,high_school_chemistry,"[HNO2, HClO4, H2S, H3PO4]",1
200,The collision theory of reaction rates does no...,high_school_chemistry,"[the number of collisions per second, the tran...",1
201,"When the dichromate ion is reacted, one of its...",high_school_chemistry,"[3+, reduction, 12+, reduction, 6+, reduction,...",2


In [6]:
# setting up prompts


from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate

# Define your customized prompt
template = """You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a cehmistry question. The Question ans been annotated in natural language. Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It's your job to fill in LLM Response
The question is: {question}
The annotation and reasoning information is:{ChEBI_Info}

The possible respones are {choices}

LLM Response:"""



custom_prompt = ChatPromptTemplate.from_template(template)

# Create an instance of the Ollama model with the customized prompt
ollama_model = Ollama(model="llama2")


question = "What are the electrical properties of elemental carbon?"


p = custom_prompt.invoke({'question':question,'ChEBI_Info':chebi})

print(p)

NameError: name 'chebi' is not defined

In [8]:
choices = 'bleh'

In [9]:
p = custom_prompt.invoke({'question':question,'ChEBI_Info':chebi,'choices':choices})

In [12]:
p.messages[0].content

'You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a cehmistry question. The Question ans been annotated in natural language. Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It\'s your job to fill in LLM Response\nThe question is: What are the electrical properties of elemental carbon?\nThe annotation and reasoning information is:The original sentence analyzed is: "What are the electrical properties of elemental carbon?"\n\nThe entities identified in this sentence are: Elemental carbon (elemental carbon).\n\nThe reasoning process followed these steps: Looking up Elemental carbon (elemental carbon) in the ChEBI ontology   - Found entity in the ontology: elemental carbon   - Looking for parent classes of Elemental carbon     - Found parent classes: elemental molecular entity, organic molecular entity   - Looking for properties of Elemental

In [7]:
from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate




# Create an instance of the Ollama model with the customized prompt
llm = Ollama(model="llama2")


In [10]:
from tqdm import tqdm
results = {}
c =0 
for i,j in tqdm(df.iterrows()):

    question =  df['question'][i]
    choices = df['choices'][i]
    info = Process_Text(question)
    ans = df['answer'][i]
    prompt = custom_prompt.invoke({'question':question,'ChEBI_Info':info,'choices':choices})

    LLM_response = llm.invoke(prompt.messages[0].content)

    c+=1
    results[c] = {
        'question': {
            'text': question,
            'choices': choices,
            'answer': ans
        },
        'OntologyInfo':info,
        'model_response': LLM_response
    }

    if c>5:
        break
    


0it [00:00, ?it/s]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'London dispersion forces are caused by', 'entities': [], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
{'text': 'London dispersion forces are caused by', 'entities': [], 'relationships': [], 'consistency_results': [], 'reasoning': {'reasoning_steps': [], 'all_facts': [], 'entity_facts': {}, 'relationship_facts': {'steps': [], 'facts': []}, 'explanation': "The statement 'London dispersion forces are caused by' is consistent with the ChEBI ontology.\n\nEntities found in the statement:\n\nRelationships found in the statement:\n\nFacts derived from the ChEBI ontology:\n- No facts could be derived from the ontology\n\n"}}
Loading ChEBI ontology from /home/matt/

1it [01:29, 89.77s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'Carbon has an atomic radius of 77 pm and a first ionization energy of 1086 kJ/mol.', 'entities': [{'name': 'Carbon', 'id': 'CHEBI:27594', 'span': (0, 6)}, {'name': 'An', 'id': 'CHEBI:33320', 'span': (11, 13)}, {'name': 'Pm', 'id': 'CHEBI:33373', 'span': (34, 36)}, {'name': 'A', 'id': 'CHEBI:13193', 'span': (41, 42)}], 'relationships': [{'subject': {'name': 'Carbon', 'id': 'CHEBI:27594'}, 'relationship': 'has_part', 'object': {'name': 'An', 'id': 'CHEBI:33320'}}], 'consistency_results': [{'relationship': {'subject': {'name': 'Carbon', 'id': 'CHEBI:27594'}, 'relationship': 'has_part', 'object': {'name': 'An', 'id': 'CHEBI:33320'}}, 'is_consistent': False, 'explanation': "No 'has_part' relationship between CHEBI_27594 and CHEBI_33320 found."}]}
✅ Successfully loaded C

2it [01:59, 54.30s/it]

...

4it [03:07, 40.73s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'The symbol for antimony is', 'entities': [{'name': 'Antimony', 'id': 'CHEBI:30513', 'span': (15, 23)}], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
{'text': 'The symbol for antimony is', 'entities': [{'name': 'Antimony', 'id': 'CHEBI:30513', 'span': (15, 23)}], 'relationships': [], 'consistency_results': [], 'reasoning': {'reasoning_steps': ['Looking up Antimony (CHEBI:30513) in the ChEBI ontology', '  - Found entity in the ontology: CHEBI_30513', '  - Looking for parent classes of Antimony', '    - Found parent classes: CHEBI_137980, CHEBI_33300', '  - Looking for properties of Antimony in annotations', '    - Found properties in annotations:', '      -

5it [03:38, 37.31s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'A sealed, rigid container contains three gases: 28.0 g of nitrogen, 40.0 g of argon, and 36.0 g of water vapor. If the total pressure exerted by the gases is 2.0 atm, what is the partial pressure of the nitrogen?', 'entities': [{'name': 'A', 'id': 'CHEBI:13193', 'span': (0, 1)}, {'name': 'G', 'id': 'CHEBI:64654', 'span': (53, 54)}, {'name': 'Nitrogen', 'id': 'CHEBI:25555', 'span': (58, 66)}, {'name': 'Argon', 'id': 'CHEBI:49475', 'span': (78, 83)}, {'name': 'Water', 'id': 'CHEBI:15377', 'span': (99, 104)}, {'name': 'If', 'id': 'CHEBI:74075', 'span': (112, 114)}], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
{'text': 'A sealed, rigid container contains thr

5it [04:10, 50.18s/it]


In [11]:
results

{1: {'question': {'text': 'London dispersion forces are caused by',
   'choices': array(['temporary dipoles created by the position of electrons around the nuclei in a molecule',
          'the three-dimensional intermolecular bonding present in all covalent substances',
          'the uneven electron-to-proton ratio found on individual atoms of a molecule',
          'the electronegativity differences between the different atoms in a molecule'],
         dtype=object),
   'answer': 0},
  'model_response': "London dispersion forces are caused by temporary dipoles created by the position of electrons around the nuclei in a molecule.\n\nExplanation:\nThe annotation provided states that London dispersion forces are caused by something, but it doesn't specify what exactly. However, based on the context and the information provided, the most plausible answer is that London dispersion forces are caused by temporary dipoles created by the position of electrons around the nuclei in a molecule.

In [25]:
results 

{'joe': 'mama', 'seph': 'stalin'}

In [38]:
llm.invoke('Answer this question: An unknown substance is found to have a high melting point. In addition, it is a poor conductor of electricity and does not dissolve in water. The substance most likely contains?. The choices are ionic bonding, nonpolar covalent bonding,covalent network bonding, metallic bonding')

"\nBased on the given properties, the substance is most likely to contain nonpolar covalent bonds. Here's why:\n\n1. High melting point: Nonpolar covalent bonds have a higher melting point compared to ionic bonds, which are held together by electrostatic forces.\n2. Poor conductor of electricity: Nonpolar covalent bonds do not allow for the easy flow of electrons, making the substance a poor conductor of electricity.\n3. Does not dissolve in water: Nonpolar molecules generally do not dissolve in polar solvents like water, as they are unable to form hydrogen bonds with water molecules.\n\nTherefore, the correct answer is nonpolar covalent bonding."

Trial setup for annotations on answer 

In [19]:
from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate

# Define your customized prompt
Initial_template = """You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a cehmistry question. The Question ans been annotated in natural language. Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It's your job to fill in LLM Response
The question is: {question}
The annotation and reasoning information is:{ChEBI_Info}

The possible respones are {choices}

LLM Response:"""

loop_template="""
You are a neuro-symbolic system being tested on your knowledge of chemistry. You are given you initial response, along with annotation and reasoning on that resonse. Use the annotation and reasoning info to give a final answer. Youa are tasked with filling in the final answer.
The original question is: {question}
The initial response was: {LLM}
The reasoning information on the intitial respose is: {LLM_Reasoning}

Your final answer is:


"""


custom_prompt = ChatPromptTemplate.from_template(template)
loop_prompt = ChatPromptTemplate.from_template(loop_template)

In [20]:
from tqdm import tqdm
results = {}
c =0 
for i,j in tqdm(df.iterrows()):

    question =  df['question'][i]
    choices = df['choices'][i]
    
    ans = df['answer'][i]

    initial = llm.invoke(question)
    info = Process_Text(initial)

    prompt = loop_prompt.invoke({'question':question,'LLM':initial,'LLM_Reasoning':info})

    LLM_response = llm.invoke(prompt.messages[0].content)



    c+=1
    results[c] = {
        'question': {
            'text': question,
            'choices': choices,
            'answer': ans
        },
        'OntologyInfo':info,
        'model_response': LLM_response
    }

    if c>5:
        break
    

0it [00:00, ?it/s]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': 'London dispersion forces are caused by the temporary dipoles that form between molecules due to their movement and the resulting distortion of the electric field. These dipoles are known as London dispersion forces, named after the British chemist Henry Edward Armstrong London, who first described them in the early 20th century.\n\nThe London dispersion force is a type of intermolecular force that arises from the movement of molecules in a liquid or gas. As the molecules move and vibrate, they create temporary dipoles that are oriented in different directions. These dipoles induce an electric field that interacts with the electric fields of nearby molecules, causing them to attract or repel each other.\n\nThe London dispersion force is weaker than other types of in

1it [00:46, 46.97s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': '\nTo determine the electronegativity of carbon, we can use the following formula:\n\nElectronegativity = (Ionization Energy / Atomic Radius) x 10^2\n\nSubstituting the values given in the question, we get:\n\nElectronegativity of Carbon = (1086 kJ/mol / 77 pm) x 10^2\n= 14.5 electronegativity units\n\nTherefore, the electronegativity of carbon is approximately 14.5.', 'entities': [{'name': 'Carbon', 'id': 'CHEBI:27594', 'span': (39, 45)}, {'name': 'We', 'id': 'CHEBI:74869', 'span': (47, 49)}, {'name': 'In', 'id': 'CHEBI:30430', 'span': (176, 178)}, {'name': 'Pm', 'id': 'CHEBI:33373', 'span': (250, 252)}, {'name': '14', 'id': 'CHEBI:28875', 'span': (263, 265)}], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.

2it [01:19, 38.66s/it]

...

4it [02:53, 43.65s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': '\nThe symbol for antimony is Sb.', 'entities': [{'name': 'Antimony', 'id': 'CHEBI:30513', 'span': (16, 24)}, {'name': 'Sb', 'id': 'CHEBI:30513', 'span': (28, 30)}], 'relationships': [], 'consistency_results': []}
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
{'text': '\nThe symbol for antimony is Sb.', 'entities': [{'name': 'Antimony', 'id': 'CHEBI:30513', 'span': (16, 24)}, {'name': 'Sb', 'id': 'CHEBI:30513', 'span': (28, 30)}], 'relationships': [], 'consistency_results': [], 'reasoning': {'reasoning_steps': ['Looking up Antimony (CHEBI:30513) in the ChEBI ontology', '  - Found entity in the ontology: CHEBI_30513', '  - Looking for parent classes of Antimony', '    - Found parent classes: CHEBI_137980,

5it [03:22, 38.41s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
{'text': "\nTo find the partial pressure of the nitrogen, we need to use the ideal gas law, which states that PV = nRT, where P is the pressure of the gas, V is its volume, n is the number of moles of gas, R is the gas constant, and T is the temperature in Kelvin.\n\nFirst, let's find the total volume of the container:\n\nV = 100 mL (since the container is sealed and rigid, its volume is fixed)\n\nNext, let's find the number of moles of each gas:\n\nn(N2) = mass of N2 / molecular weight of N2 = 28.0 g / 28.97 g/mol = 0.96 mol\nn(Ar) = mass of Ar / molecular weight of Ar = 40.0 g / 39.94 g/mol = 1.01 mol\nn(H2O) = mass of H2O / molecular weight of H2O = 36.0 g / 18.02 g/mol = 2.0 mol\n\nNow, let's use the ideal gas law to find the partial pressure of the nitrogen:\n\nP(N2) = 

5it [04:06, 49.29s/it]


In [21]:
results

{1: {'question': {'text': 'London dispersion forces are caused by',
   'choices': array(['temporary dipoles created by the position of electrons around the nuclei in a molecule',
          'the three-dimensional intermolecular bonding present in all covalent substances',
          'the uneven electron-to-proton ratio found on individual atoms of a molecule',
          'the electronegativity differences between the different atoms in a molecule'],
         dtype=object),
   'answer': 0},
  'OntologyInfo': 'The original sentence analyzed is: "London dispersion forces are caused by the temporary dipoles that form between molecules due to their movement and the resulting distortion of the electric field. These dipoles are known as London dispersion forces, named after the British chemist Henry Edward Armstrong London, who first described them in the early 20th century.\n\nThe London dispersion force is a type of intermolecular force that arises from the movement of molecules in a liquid or

: 

In [17]:
thinco

'London dispersion forces are a type of intermolecular force that arises due to the temporary dipoles that form in molecules as they vibrate. These forces are also known as London dispersion forces or Van der Waals forces, and they are responsible for the attractive interactions between molecules in a liquid or gas.\n\nLondon dispersion forces are caused by the temporary dipoles that form in molecules as they vibrate. When a molecule vibrates, its electrons move back and forth around the nucleus, creating an instantaneous dipole moment. This dipole moment is temporary because the electrons quickly return to their equilibrium position around the nucleus. However, during this time, the molecule is attracted to other molecules with opposite charge, resulting in a weak intermolecular force.\n\nThese forces are weaker than covalent bonds and ionic bonds but stronger than hydrogen bonds. They are responsible for the cohesion of liquids and gases, and they play a crucial role in determining t

In [2]:
import sys
import os 
import json 
import gc  # For garbage collection
import time
import numpy as np
from tqdm import tqdm  # For progress tracking
sys.path.append('../../src')
from pipeline.ChEBIPipeline import ChEBIPipeline as AnnotationCheck
from reasoning.OntoValidation.ChEBIReasoner import ChEBIReasoner, reason_with_chebi
from reasoning.LLMFeedback.Pipe2NL import extract_chebi_ids, ChEBINameRetriever, replace_chebi_ids, format_json_as_prompt
from langchain_community.llms import Ollama
from langchain.prompts import ChatPromptTemplate
import pandas as pd

# Paths
BASE_PATH = '/home/matt/Proj/Hermeticav2'
OUTPUT_DIR = f"{BASE_PATH}/results/QAnnotationOnly"
OUTPUT_FILE = f"{OUTPUT_DIR}/MMLU_highschool_Chem.json"

# Ensure directories exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Custom JSON encoder to handle numpy arrays and other non-serializable types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if pd.isna(obj):
            return None
        return super(NumpyEncoder, self).default(obj)

def Process_Text(text):
    # start with annotations
    annotator = AnnotationCheck(f'{BASE_PATH}/notebooks/prototyping/QAReasoning/chebi.obo',
                               f'{BASE_PATH}/data/ontologies/Chemistry/chebi.owl')
    annotations = annotator.process_text(text)
    Reasoned = reason_with_chebi(annotations, f'{BASE_PATH}/data/ontologies/Chemistry/chebi.owl')
    
    # convert the stuff back into NL for LLM input 
    finder = ChEBINameRetriever(f'{BASE_PATH}/data/ontologies/Chemistry/chebi.owl')
    IDs = extract_chebi_ids(Reasoned)
    mapping = finder.get_chebi_names(IDs)
    PreNL = replace_chebi_ids(Reasoned, mapping)
    NL = format_json_as_prompt(PreNL)
    
    # Force garbage collection after heavy processing
    gc.collect()
    
    return NL

# Define your customized prompts
template = """You are a neuro-symbolic system being tested on your knowledge of chemistry. You are tasked with answering a multiple choice chemistry question.
The Question ans been annotated in natural language.
Use the annotation and reasoning information to aid your response to the question. Ignore errors in the annotation and reasoning information. It's your job to fill in the Final Answer.
You are given a set of possible answers to choose from.
Make sure you answer with ONLY the correct reponse.

The question is: {question}
The annotation and reasoning information is:{ChEBI_Info}

The possible respones are {choices}

Final Answer:"""

template2 = """
You are a chemsitry expert being tested on your knowledge of chemsitry. You are given a multiple choice question and a set of possible answers. Make sure you respond ONLY with the correct answer.

The question is {question}

The possible responses are {choices}

Final Answer:

"""

custom_prompt = ChatPromptTemplate.from_template(template)
custom_dummy = ChatPromptTemplate.from_template(template2)

# Function to load existing results if available
def load_existing_results():
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                print(f"Error loading existing results file. Starting fresh.")
    return {}

# Function to save results after each individual question
def save_result(results, key, data):
    # Add new data to results
    results[key] = data
    
    # Save the entire results file
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)
    
    print(f"Updated results file after processing {key}")
    return results

# Function to convert any pandas/numpy types to Python native types
def convert_to_serializable(obj):
    if isinstance(obj, (np.ndarray, pd.Series)):
        return obj.tolist()
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict(orient='records')
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    else:
        return obj

# Main processing function with individual saving
def main():
    # Load data
    df = pd.read_parquet(f'{BASE_PATH}/data/Raw/MMLU/mmlu/high_school_chemistry/test-00000-of-00001.parquet')
    llm_list = ['llama2', 'llama3.1', 'llama3.2', 'phi', 'phi3', 'phi3.5', 'phi4', 'deepseek-r1:14b']
    
    # Load or initialize results
    results = load_existing_results()
    
    # Process each model and question
    for model in llm_list:
        print(f"Processing model: {model}")
        llm = Ollama(model=model)
        
        # Check which questions have already been processed for this model
        processed_indices = set()
        for key in results.keys():
            if key.startswith(f"{model}_"):
                try:
                    idx = int(key.split('_')[1])
                    processed_indices.add(idx)
                except (ValueError, IndexError):
                    pass
        
        # Process each question individually
        for i in tqdm(range(len(df))):
            if i in processed_indices:
                print(f"Skipping already processed question {i} for model {model}")
                continue
                
            try:
                # Convert pandas series elements to Python native types
                question = str(df['question'][i])
                choices = convert_to_serializable(df['choices'][i])
                ans = str(df['answer'][i]) if not pd.isna(df['answer'][i]) else None
                
                # Process text and get responses
                info = Process_Text(question)
                
                # Make sure info is JSON serializable
                info = convert_to_serializable(info)
                
                dummyPrompt = custom_dummy.invoke({'question': question, 'choices': choices})
                Rawresponse = llm.invoke(dummyPrompt.messages[0].content)
                
                prompt = custom_prompt.invoke({'question': question, 'ChEBI_Info': info, 'choices': choices})
                LLM_response = llm.invoke(prompt.messages[0].content)
                
                # Create result data
                data = {
                    'question': {
                        'text': question,
                        'choices': choices,
                        'answer': ans
                    },
                    'OntologyInfo': info,
                    'onto_model_response': LLM_response,
                    'raw_model_response': Rawresponse
                }
                
                # Save immediately after each question
                results = save_result(results, f'{model}_{i}', data)
                
                # Free up memory
                del info, prompt, dummyPrompt, data
                gc.collect()
                
            except Exception as e:
                print(f"Error processing question {i} for model {model}: {str(e)}")
                # Continue with next question despite errors
            
            # Brief pause to let system recover
            time.sleep(0.5)
    
    print(f"All processing complete. Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Processing model: llama2


  0%|          | 0/203 [00:00<?, ?it/s]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Loading ChEBI ontology from /home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl...
Ontology loaded.
Pre-caching entity labels...
Cached 202209 entity labels.
Updated results file after processing llama2_0


  0%|          | 1/203 [00:52<2:55:06, 52.01s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Loading ChEBI ontology from /home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl...
Ontology loaded.
Pre-caching entity labels...
Cached 202209 entity labels.
Updated results file after processing llama2_1


...

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.
✅ Loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 entries.
Pipeline initialization complete.
✅ Successfully loaded ChEBI ontology with 220816 classes.
✅ Built entity cache with 220816 mapped ChEBI IDs.
Loading ChEBI ontology from /home/matt/Proj/Hermeticav2/data/ontologies/Chemistry/chebi.owl...
Ontology loaded.
Pre-caching entity labels...
Cached 202209 entity labels.
Updated results file after processing llama2_71


 35%|███▌      | 72/203 [1:02:38<1:55:01, 52.68s/it]

Initializing ChEBI pipeline components...
✅ Loaded ChEBI ontology: 539620 entities extracted.


 35%|███▌      | 72/203 [1:02:46<1:54:12, 52.31s/it]


KeyboardInterrupt: 